# IV H-scan Asymmetric Component Batch Analysis

This notebook analyzes the **asymmetric component** of IV curves from magnetic field hysteresis scans using Gaussian filter decomposition.

**Analysis Focus:**
- Asymmetric current component: $I_{asym}(V) = \frac{I(V) - I(-V)}{2}$
- Odd function property: $I_{asym}(-V) = -I_{asym}(V)$
- Reveals rectification and asymmetric transport effects

**Data Source:** Pre-processed DataFrame with Gaussian filter decomposition

In [ ]:
# Notebook setup - enables autoreload, imports, and path management
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Import batch processing modules
from scripts.IV_Hscan_gaussian import (
    process_folder,
    parse_folder_name,
    build_dataframe,
    save_dataframe,
    load_dataframe,
    get_asymmetric_current_at_voltage
)

## 1. Process Folder - Batch Gaussian Filter Analysis

In [ ]:
# Define the following three paramters before running the analysis

#only change the folder name with new folder name
folder_name = "20251117113423 b_scan_100K__92steps_H0.90T_phi_84.0to84.0_theta_0.0to0.0°_v0.datfolder"
#write the scan type here
scan_type = "b_scans"
#enter the correct temerature manually
temperature = 100  # K
folder_path = PROJECT_ROOT / "data" / "device 2" / scan_type / "IV_H_scans" / folder_name

print(f"Processing folder: {folder_path}")
print(f"Folder exists: {folder_path.exists()}")

In [ ]:
# Process all IV scans with Gaussian filter
# This will:
#  - Load each .dat file
#  - Apply Gaussian filter with sigma=1.5
#  - Calculate symmetric/asymmetric decomposition
#  - Return dictionary of results

filtered_dict, metadata = process_folder(
    folder_path=str(folder_path),
    sigma=1.5,
    verbose=True
)

print(f"\nProcessed {len(filtered_dict)} IV scans successfully")

## 2. Create DataFrame with Asymmetric Decomposition

In [ ]:
# Extract temperature from metadata
print(f"Temperature: {temperature} K")

# Build DataFrame (includes raw data and symmetric/asymmetric components)
df = build_dataframe(filtered_dict, temperature=temperature)

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Display first few rows (scalars only)
scalar_cols = ['timestamp', 'H', 'Hx', 'Hy', 'Hz', 'sigma', 'v_offset', 'residual_rms']
df[scalar_cols].head(10)

In [ ]:
# Save DataFrame to pickle (preserves arrays)
output_filename = f"IV_gaussian_{temperature}K"
output_path = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / scan_type / output_filename

save_dataframe(df, output_path, format='pickle', save_csv_scalars=True)

print(f"\nDataFrame saved!")
print(f"To reload: df = load_dataframe('{output_path}.pkl')")

## 3. Verify Asymmetric Component Properties

Check that asymmetric component behaves as odd function: $I_{asym}(-V) = -I_{asym}(V)$

## 4. I_asym(H) Curves at Fixed Voltages

Asymmetric magnetoresistance: how the asymmetric transport component changes with magnetic field.

In [ ]:
# USER CONFIGURATION: Select voltages to analyze
voltages_to_plot = [-0.4, -0.2, 0.0, 0.2, 0.4]  # Volts

print(f"Will plot I_asym(H) curves at {len(voltages_to_plot)} voltages:")
print(voltages_to_plot)

# Extract I_asym(H) data at each voltage
fig, ax = plt.subplots(figsize=(12, 7))

# Use a color map for different voltages
colors = plt.cm.RdBu_r(np.linspace(0.1, 0.9, len(voltages_to_plot)))

for V_val, color in zip(voltages_to_plot, colors):
    # Get asymmetric current at this voltage
    I_asym_vs_H = get_asymmetric_current_at_voltage(df, V_val)
    
    ax.plot(I_asym_vs_H['H'], I_asym_vs_H['I_asym_at_V'] * 1e6, 'o-', 
            color=color, markersize=4, linewidth=2, 
            label=f'V = {V_val:+.2f} V', alpha=0.8)

ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('$I_{asym}$ (µA)', fontsize=14)
ax.set_title(f'Asymmetric Current vs Magnetic Field at Fixed Voltages\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
ax.legend(fontsize=10, loc='best', ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 2D Color Map: I_asym(H, V)

Visualize asymmetric current across the full H-V parameter space.

In [ ]:
print("="*70)
print("ASYMMETRIC COMPONENT BATCH ANALYSIS COMPLETE")
print("="*70)
print(f"\nFolder: {folder_name}")
print(f"Temperature: {temperature} K")
print(f"Total IV scans processed: {len(df)}")
print(f"H field range: {df['H'].min():.4f} to {df['H'].max():.4f} T")
print(f"\nGaussian filter sigma: {df['sigma'].iloc[0]}")
print(f"Mean voltage offset: {df['v_offset'].mean():.6f} V")
print(f"Mean residual RMS: {df['residual_rms'].mean():.4e} A")
print(f"\nResults saved to:")
print(f"  - DataFrame: {output_path}.pkl")
print(f"  - Scalars CSV: {output_path}.csv")
print("="*70)

## 7. Export Asymmetric Analysis Results